In [ ]:
/*
================================================================================
  HANDLING HIGH READ TRAFFIC — EXPLAINED LIKE WE'RE JUST TALKING ABOUT IT
================================================================================

  Here's the situation that comes up in basically every popular app:

      A FEW people create stuff (post a photo, list a product, upload
      a video)... but a HUGE number of people just come to LOOK at it.

  Think Instagram: one influencer posts a photo, and a million people
  view it. That's a "read-to-write ratio" of like 1,000,000 : 1.
  Almost nobody is "writing" (posting). EVERYONE is "reading" (viewing).

  ANALOGY: Think of a popular YouTube video.
    - The creator uploads the video ONCE (that's the "write")
    - Ten million people click play and watch it (that's the "read")

  If every single one of those 10 million views had to go all the way
  back to some giant original hard drive and re-read the raw video file
  from scratch every time... that hard drive would catch fire. Literally
  melt under the load.

  So the WHOLE POINT of this topic is:

      "How do we serve all those repeated reads WITHOUT hammering our
       main database (or file, or origin server) every single time?"

  The answer is basically: don't make the "expensive, single source of
  truth" thing answer every question. Put cheaper, faster copies of the
  answer in front of it, and only go to the expensive thing when you
  really have to.


--------------------------------------------------------------------------------
  1. WHY IS THIS ACTUALLY HARD? (3 separate problems hiding in one)
--------------------------------------------------------------------------------

  It's tempting to think "just get a bigger database!" But there are
  really THREE separate problems bundled together here:

  PROBLEM 1: SCALE (there's just... a LOT of requests)

    Imagine: 100 million people use your app, each checks their feed
    10 times a day. That's 1 BILLION feed views... per day.
    Spread across 24 hours, that's roughly 11,500 requests EVERY SECOND
    on average — and during peak hours (everyone checking at lunch, say)
    it can spike to 35,000+ requests per second.

    ANALOGY: Imagine one cashier trying to check out every single
    customer at every Walmart in the country, all by themselves.
    Doesn't matter how fast that one cashier is — the sheer NUMBER of
    people is the problem, not the cashier's skill.

  PROBLEM 2: LATENCY (people expect answers FAST)

    Even if your database COULD handle all that volume, people expect
    a page to load in well under a second. Some expectations:

        Autocomplete (as-you-type search): < 50ms
        API response:                      < 100ms
        Page load:                         < 200ms
        Search results:                    < 300ms

    But a typical database query, once you count network trips, disk
    reads, and processing, might take anywhere from 50ms to 2 FULL
    SECONDS. That gap is a real problem, not just a "nice to have."

  PROBLEM 3: COST (databases are EXPENSIVE to scale)

    Beefing up your main database (more CPU, more memory, more
    replication, more backups) is one of the most expensive ways to
    handle more traffic. Serving the same data from a cheap, fast
    memory cache is WAY cheaper per request.

  ANALOGY for all 3 combined: Running one single, giant, fancy library
  where every visitor has to walk up to the one head librarian and ask
  her personally to go find their book in the back room. She's smart,
  she's fast, but she can only help ONE person at a time, and there's a
  huge line snaking out the door.

  The fix isn't "get a faster librarian." The fix is: put copies of the
  popular books out on shelves near the front door so people don't need
  the librarian for those at all.


--------------------------------------------------------------------------------
  2. WHY ARE SO MANY REAL APPS LIKE THIS?
--------------------------------------------------------------------------------

  System                | Rough Read : Write Ratio | Why
  ------------------------|---------------------------|--------------------------------
  Social media feed       | 1,000 : 1                 | Tons of viewers, few posters
  E-commerce catalog      | 10,000 : 1                | Millions browse, few actually buy
  News website            | 100,000 : 1               | Article written once, read forever
  Video streaming         | 10,000 : 1                | Upload once, streamed millions of times
  URL shortener           | 10,000 : 1                | Create once, redirected to constantly

  These numbers aren't exact science, but the SHAPE is always the same:
  "created once, consumed a LOT."

  And that repetition is actually GOOD NEWS — it means:
    - If the same data gets read over and over → cache it!
    - If it's okay for data to be a little "old" → use a TTL (auto-expire)
    - If your users are spread around the world → move data physically
      closer to them (CDN)

  Before reaching for any tool, ask yourself:

    Question                                    | Why it matters
    ----------------------------------------------|----------------------------------
    What's the read:write ratio?                  | Tells you if reads are even the bottleneck
    Are reads REPEATED or mostly UNIQUE?           | Repeats = cache. Unique = better indexes/search
    How fresh does data need to be?                | Determines if staleness is even allowed
    Is this data public or user-specific/private?  | Public = cache aggressively. Private = be careful
    What happens on a cache miss?                  | The database still needs to survive this
    Are there "hot" objects (viral posts, etc)?     | Need special handling (more on this later)
    Are your users global?                          | You probably need a CDN


--------------------------------------------------------------------------------
  3. THE TOOLKIT — YOUR MAIN WEAPONS AGAINST HIGH READ TRAFFIC
--------------------------------------------------------------------------------

  You've got 6 main tools. You'll almost always combine several of them,
  not pick just one.

    Tool                    | What it does                          | Rough impact
    --------------------------|----------------------------------------|---------------------------
    Caching                  | Store popular data somewhere faster    | Huge — often the #1 lever
    CDN / Edge                | Serve cacheable stuff near the user    | Big win for global latency
    Read Replicas             | Copies of the DB that only handle reads| More database read capacity
    Database Optimization     | Indexes, better queries                | Makes remaining DB reads fast
    Pre-computation            | Calculate answers BEFORE they're asked | Fast reads for expensive stuff
    Load Balancing             | Spread traffic across many servers     | Avoids overloading one machine

  GENERAL ORDER OF ATTACK (a good default, not a strict law):
    1. Cache repeated reads
    2. Use a CDN for anything cacheable and public
    3. Add read replicas for what's left hitting the database
    4. Optimize the database queries themselves (indexes etc)
    5. Pre-compute the genuinely expensive stuff


--------------------------------------------------------------------------------
  4. CACHING — THE FIRST LINE OF DEFENSE (and usually the biggest win)
--------------------------------------------------------------------------------

  ANALOGY: Sticky notes on your fridge. Instead of digging through a
  filing cabinet in the basement every time you need your grocery list,
  you keep a copy stuck on the fridge where it's instantly visible.
  That's a cache — a fast, nearby copy of something that lives somewhere
  slower.

  4.1 THE CACHING HIERARCHY (multiple layers, like a funnel)

    Every layer below catches some requests so they NEVER reach the
    next (slower) layer:

        User Request
             |
             v
       +-------------+   0ms - right there on your device
       | Browser Cache|
       +------+-------+
              v
       +-------------+   10-50ms - nearest edge server
       |  CDN Cache   |
       +------+-------+
              v
       +-------------+
       | Load Balancer|
       +------+-------+
              v
       +-------------+
       |  App Server  |
       +------+-------+
              v
       +---------------------+  1-5ms - Redis / Memcached
       | Application Cache    |
       +----------+-----------+
                  v
       +---------------------+  0.1-1ms, but tiny/limited
       | DB Buffer Pool        |
       +----------+-----------+
                  v
       +-------------+   The slow one - actual disk reads
       |  Database    |
       +-------------+

  The GOAL is to answer as many requests as possible using the TOP
  layers, and only let the truly necessary requests trickle down to
  the database at the bottom.

  4.2 CACHE-ASIDE PATTERN (the most common approach — "lazy loading")

    Here's how a typical read works with a cache in front of your DB:

        Client        App Server        Redis Cache       Database
          |                |                  |               |
          |--GET /user/123->|                  |               |
          |                |--GET user:123--->|               |
          |                |                  |
          |      [CACHE HIT]  ->  return in ~2ms, done!
          |
          |                |                  |  [CACHE MISS - key not found]
          |                |--SELECT * FROM users WHERE id=123-------------->|
          |                |<----------------user data----------------------|
          |                |--SET user:123 (with a TTL / expiry time)------>|
          |<--return data--|

    In plain English:
      1. Check the cache first
      2. Found it? Great, return it — fast!
      3. Not found? Go to the database, get it, THEN save a copy
         in the cache for next time

    Why this matters: if 90% of requests are found in the cache (a
    "90% hit rate"), your database load drops by roughly 10x for that
    endpoint. That's the number that matters in interviews — not just
    "we have a cache," but "what's our hit rate?"

  4.3 KEEPING THE CACHE FRESH — INVALIDATION STRATEGIES

    The cache is a COPY. When the real data changes, the copy can
    become stale/wrong. You need a plan for this. Three common ones:

    STRATEGY 1 — TTL (Time To Live): "Just expire it after X minutes"

        Write to DB -> Data sits in Cache -> TTL counts down ->
        Cache Expires -> Next Read = Miss -> Refresh from DB

      Simplest option. A 5-minute TTL means data could be up to 5
      minutes stale. Totally fine for a news feed. NOT fine for your
      bank account balance.

    STRATEGY 2 — Write-Through: "Update both at the same time"

        Client -> App Server -> updates Database AND Cache together,
        right in the same write operation, before saying "success"

      Keeps things fresher, but writes get a little slower, and you
      have to decide: "what if the DB write succeeds but the cache
      write fails?"

    STRATEGY 3 — Event-Based Invalidation: "Announce the change, let
    everyone clean up their own copy"

        Write to DB -> Emit an Event -> Message Queue (Kafka etc) ->
        Multiple services each hear the event -> each deletes/updates
        their OWN cached copy

      More moving parts, but very flexible — great when many different
      services each keep their own cache.

    QUICK COMPARISON:

      Pattern            | Freshness | Write speed        | Complexity | Best for
      --------------------|-----------|---------------------|------------|---------------------------
      Cache-aside + TTL   | Eventual  | Fast (low overhead) | Low        | Most apps — feeds, catalogs
      Write-through       | Strong-ish| Slower              | Medium     | Small critical data, can't tolerate stale
      Event-based         | Eventual  | Fast                | High       | Microservices, many cache layers

    Default advice: start with cache-aside + TTL. Only add complexity
    (write-through, events) when the business genuinely needs it.

  4.4 WHAT SHOULD YOU ACTUALLY CACHE?

    Good candidates                    | Bad candidates
    -------------------------------------|--------------------------------------
    User profiles                       | Rapidly-changing live prices/quotes
    Product details                     | Bank account balances
    News feed content                   | One-time-use auth tokens
    Search results                      | Highly personalized, unique-per-user data
    Configuration/settings data          | Huge binary files (use a CDN instead)
    Session data                        | Data almost nobody ever reads

    Rule of thumb: cache things that are READ A LOT and CHANGE
    SLOWLY. If something changes every second and must always be
    exactly correct, caching it is risky.

  4.5 THE SCARY PART: HOT KEYS & CACHE STAMPEDE

    ANALOGY: Imagine one single ice cream shop that EVERYONE in town
    suddenly wants to visit at the exact same minute, because a
    celebrity just tweeted about it. That's a "hot key" — one specific
    item getting a disproportionate amount of traffic.

    THE HOT-KEY PROBLEM:
      In a cache spread across multiple servers, each individual "key"
      (like "post:12345") lives on exactly ONE server/node. If that ONE
      key suddenly gets a massive share of ALL your traffic (viral
      post, flash sale item), that ONE node gets slammed while all your
      OTHER cache nodes sit around doing nothing. Adding more nodes
      doesn't help — the problem is concentrated on ONE key, not spread out.

    CACHE STAMPEDE (a.k.a. "thundering herd"):
      This happens the MOMENT a hot key expires.

          Cache          Requests (tons, all at once)          Database
            |                       |                              |
            | [hot key just expired]|                              |
            |<---GET hot:key--------|  (thousands of requests, same instant)
            |---MISS (all of them)->|
                                    |--ALL of them query the DB at once-->|
                                    |        (database instantly overloaded)

      While the value was cached, the database was basically getting a
      free vacation for that key. The SECOND it expires, EVERY pending
      request notices it's missing at the same time and ALL of them run
      to the database simultaneously to recompute the exact same thing.
      Your database can go from "chill" to "on fire" in a split second.

    FIXES (these combine well together):

      Technique                        | How it works                                          | Fixes
      -------------------------------- | ------------------------------------------------------ | -----------
      Request coalescing (single-flight)| Only the FIRST request recomputes; everyone else waits | Stampede
      Locking + stale fallback          | Lock briefly to refresh; others get a slightly old value| Stampede
      Probabilistic early expiration    | Randomly refresh a bit BEFORE expiry, spread out timing | Stampede
      Hot-key replication               | Copy the popular key onto multiple nodes                | Hot key
      Local in-process caching          | Cache the hottest stuff right in each app server's RAM  | Hot key

      Simple way to think about "request coalescing": if 1,000 people
      ask the SAME question at the SAME time, only ONE of them actually
      goes and gets the answer — the other 999 just wait for that one
      answer instead of all separately doing the same work.

    TWO MORE CASES WORTH KNOWING:

      Negative caching: if someone asks for something that DOESN'T
      exist (like a deleted post), cache that "not found" result too,
      briefly. Otherwise repeated requests for missing stuff bypass the
      cache every single time and slam the database — a cheap way for
      someone (or a bug, or an attacker) to overload you.

      Cache failure fallback: what happens if your ENTIRE cache goes
      down or restarts empty? If every request suddenly falls straight
      through to the database, the database gets hit with load it was
      never sized for. Request coalescing and small local caches help
      it survive that moment.

    It's worth mentioning hot keys/stampede in an interview even if
    nobody asks — it's one of the most common blind spots in an
    otherwise solid caching design.


--------------------------------------------------------------------------------
  5. READ REPLICAS — MORE DATABASE MUSCLE FOR READS
--------------------------------------------------------------------------------

  ANALOGY: Instead of one overworked librarian, hire several MORE
  librarians who each have an up-to-date (but slightly delayed) copy of
  every book. People needing to just LOOK something up can ask any of
  them. But if you need to add a NEW book to the library, there's still
  only ONE official master copy that gets updated first.

  HOW IT WORKS:
    - One PRIMARY database handles all WRITES
    - One or more REPLICAS constantly copy changes FROM the primary
    - READ-ONLY queries get spread across the replicas

        App Server 1 -+
        App Server 2 -+--> Proxy/Router --writes--> Primary (writes)
        App Server N -+                    |              |
                                            |        (replication stream)
                                            |              v
                                        reads---> Replica 1, Replica 2, ... Replica N

  IMPORTANT CAVEAT: replicas help when load is SPREAD across many
  different rows/keys. If ONE row/key is getting hammered (a viral
  post), every replica has the SAME hot row — replicas don't fix that,
  caching and hot-key techniques do.

  5.1 TWO WAYS TO REPLICATE DATA

    ASYNC REPLICATION (most common):

        Client -> Primary: "INSERT INTO orders..."
        Primary writes to its own disk, THEN immediately says "OK, done!"
        Primary replicates the change to the Replica AFTERWARD (could take
        10ms to 1 second)

      Pros: fast writes, replica still handles read traffic
      Cons: replica might briefly show OLD data (replication lag);
            if the primary crashes before replicating, that change
            could be lost

    SYNCHRONOUS REPLICATION:

        Client -> Primary: "INSERT INTO orders..."
        Primary writes AND waits for the replica to confirm receiving
        it BEFORE saying "OK, done!" to the client

      Pros: much safer, less risk of losing data on failover
      Cons: writes are slower; if the replica is unhealthy, it can
            even block writes entirely

    Most systems use ASYNC replication and just design around the
    occasional stale read. Synchronous is reserved for data where
    losing even a moment's worth of writes would be a real problem.

  5.2 THE ANNOYING SIDE EFFECT: REPLICATION LAG

    Classic bug scenario: you update your profile, immediately refresh
    the page, and see the OLD info. Why? Your write went to the
    PRIMARY, but your refresh's read got routed to a REPLICA that
    hasn't caught up yet.

    FIXES:
      - Read-your-writes: right after someone writes, route THEIR
        reads to the primary for a short window, then go back to normal
      - Monotonic reads: keep a user "stuck" to the same replica (or
        one that's at least as fresh as before) so their data doesn't
        appear to go "backward in time"
      - Lag monitoring: keep an eye on how far behind each replica is,
        and stop sending reads to any replica that's fallen too far behind

      Solution           | How it works                    | Trade-off
      ---------------------|----------------------------------|----------------------------
      Read-your-writes     | Recent writers read from primary | Primary gets extra load
      Monotonic reads      | Sticky session to same replica    | Uneven load across replicas
      Lag monitoring       | Only use "fresh enough" replicas  | Fewer usable replicas at times


--------------------------------------------------------------------------------
  6. CDN — SERVING CONTENT CLOSE TO PEOPLE, PHYSICALLY
--------------------------------------------------------------------------------

  ANALOGY: Instead of shipping every single online order from ONE
  warehouse in Ohio to customers all over the planet, Amazon has
  warehouses scattered around the world, so your package comes from
  somewhere nearby. Same idea, but for internet data.

  A Content Delivery Network (CDN) keeps copies of your content at
  "edge locations" scattered around the globe, so users get served
  from somewhere close to THEM, instead of your one origin server far away.

        User in Tokyo    --> Edge Server (Tokyo)    ~20ms  (cache HIT)
        User in London   --> Edge Server (London)   ~20ms  (cache HIT)
        User in New York --> Edge Server (New York) ~20ms  (cache HIT)

        (only on a MISS does it travel all the way to)
                              Origin Server (San Francisco)  80-200ms

  If the edge server already has what the user wants, the request NEVER
  has to travel all the way back to your original server. Massive
  latency win, and it also takes load OFF your origin entirely.

  6.1 WHAT SHOULD LIVE ON A CDN?

    Content type          | Suggested cache time | Notes
    -----------------------|------------------------|----------------------------------
    Static images           | 1 year                 | Rarely changes once published
    CSS/JS (versioned)      | 1 year                 | New version = new filename
    HTML pages              | 5-60 minutes           | Balance freshness vs. speed
    API responses           | 1-5 minutes            | Only if staleness is acceptable
    User-specific data       | DON'T cache at CDN      | Private — must not be shared

    HANDY TRICK — "content addressing": name your file something like
    "app.abc123.js" where "abc123" is a hash of the file's content. If
    the content changes, the filename changes too, so you can safely
    cache the OLD file forever (it'll just never be requested again)
    and the NEW file gets its own fresh cache entry automatically.

  6.2 A COOL TRICK: "stale-while-revalidate"

    This tells the CDN: "It's fine to serve the slightly-expired
    version RIGHT NOW while you go fetch the fresh version in the
    background for next time." Users always get a fast response, and
    only the very next request after expiry might be a tiny bit stale.

  6.3 EDGE COMPUTING (running small bits of code AT the edge)

    Modern CDNs can run little snippets of logic right at the edge
    server — for stuff like authentication checks, redirecting based
    on location, or A/B testing — without needing to go all the way
    back to your main servers for every little decision.

    Watch out: if every single response becomes "unique per user"
    because of heavy personalization, you lose most of the benefit of
    caching. Try to keep the CACHEABLE public part separate from the
    small PERSONALIZED overlay.

  6.4 WHEN CONTENT CHANGES — HOW DO YOU UPDATE THE CDN?

    Method            | How it works
    --------------------|--------------------------------------------------
    TTL expiration      | Just wait for it to naturally expire (simple, coarse)
    Purge by URL         | Manually tell the CDN "throw away THIS specific URL"
    Purge by tag         | Tag related content, invalidate a whole group at once
    Versioned URLs        | Cleanest — just change the filename when content changes


--------------------------------------------------------------------------------
  7. DATABASE OPTIMIZATION — MAKING THE REMAINING QUERIES FAST
--------------------------------------------------------------------------------

  Even with caching, CDN, and replicas, SOME requests will still hit
  your actual database. Let's make sure those are fast.

  7.1 INDEXES — the single biggest database speed lever

    ANALOGY: A book with NO index. If you want to find every mention of
    "dragons," you'd have to read every single page. A book WITH an
    index lets you jump straight to the right pages.

        WITHOUT INDEX:
          SELECT * FROM orders WHERE customer_id = 123
          -> scans all 10,000,000 rows one by one -> ~5 seconds (slow)

        WITH INDEX on customer_id:
          Same query
          -> jumps straight to the ~150 matching rows -> ~5 milliseconds (fast)

    For queries filtering on MULTIPLE columns, a "composite index"
    (an index built on more than one column together) helps even more.
    Order matters: put your FILTER columns first, then SORT columns.

  7.2 DENORMALIZATION — trading storage for speed

    "Normalized" databases avoid duplicating data, but that means you
    often need JOINs (combining multiple tables) to get a full picture
    — and JOINs get expensive at scale.

    "Denormalized" means you deliberately DUPLICATE some data so you
    can read it all from ONE table, with no joins needed.

        NORMALIZED (needs a 3-way JOIN every read):
          Orders Table <-join-> Customers Table <-join-> Products Table

        DENORMALIZED (one flat table, read directly):
          Orders_Denormalized [ customer_name, customer_email,
                                 product_name, product_price, ... ]

    Aspect       | Normalized                  | Denormalized
    --------------|-------------------------------|--------------------------------
    Reads         | Join-heavy                    | Simple, single-table read
    Writes        | One clean canonical copy       | Must update/rebuild duplicates
    Storage       | Less redundant                 | More redundant (uses more space)
    Consistency   | Enforced naturally at source     | You have to maintain it carefully
    Best for      | Write-heavy systems (OLTP)       | Read-heavy, analytics-style views

  7.3 GENERAL QUERY PROBLEMS AND FIXES

    Problem                  | Fix
    ----------------------------|--------------------------------------------
    Full table scan             | Add the right index
    Too many JOINs               | Denormalize, or use a materialized view
    Huge result sets              | Add LIMIT + pagination
    "N+1 queries" (1 query per   | Batch it into a single JOIN or bulk fetch
     item in a loop — a classic  |
     beginner mistake)           |
    Expensive aggregations        | Pre-compute it ahead of time (next section!)


--------------------------------------------------------------------------------
  8. PRE-COMPUTATION — DOING THE HARD WORK BEFORE ANYONE ASKS
--------------------------------------------------------------------------------

  ANALOGY: Instead of a restaurant cooking your entire meal from raw
  ingredients the SECOND you order (slow), a smart kitchen preps common
  components (chopped veggies, pre-made sauces) ahead of time, so
  assembling your dish at order-time is fast.

  8.1 MATERIALIZED VIEWS

    A materialized view is just: "run this expensive query ONCE, save
    the RESULT as if it were its own table, and just read that saved
    result instead of recalculating every time."

    Trade-off: the saved result is only as fresh as the last time you
    refreshed it — unless you set up automatic incremental updates.

    HEADS UP: a plain refresh of a materialized view often LOCKS it,
    blocking reads while it rebuilds — which defeats the whole purpose
    in a read-heavy system! Many databases offer a "concurrent refresh"
    option that avoids locking readers out (usually a bit slower to run,
    and needs a unique index set up first).

  8.2 SPECIALIZED PRE-COMPUTED DATA STORES

    Sometimes you don't just precompute a view — you use an entirely
    different, purpose-built database for certain kinds of reads:

        Primary Database
              |
              v
        Batch Job (hourly/daily) --computes/transforms-->
              |
      +-------+----------------+
      v       v                v
    Redis   Elasticsearch   ClickHouse
   (fast    (full-text      (analytics /
   lookups)  search)         event data)

    - Redis -> great for quick key-value lookups, leaderboards
    - Elasticsearch -> great for full-text search
    - ClickHouse / Druid -> great for big analytical queries

  8.3 FEED PRE-GENERATION (this is literally the "fanout" pattern!)

    Instead of building someone's feed the MOMENT they open the app,
    you build it in the BACKGROUND the moment content is POSTED:

        User Posts -> Background Worker -> generates a feed entry for
        EVERY follower, ahead of time -> stored in Redis

        Later: User Opens App -> just READ the already-built feed -> fast!

    (This is the exact same idea covered in the Fanout Pattern topic —
    it's really just "pre-computation" applied specifically to feeds.
    And just like there, celebrities/high-follower accounts usually
    need a hybrid approach instead of eager pre-computation for
    everyone.)


--------------------------------------------------------------------------------
  9. LOAD BALANCING AND CONNECTION POOLING
--------------------------------------------------------------------------------

  ANALOGY: A grocery store with 10 checkout lanes and a person at the
  front directing you to whichever lane is shortest, instead of
  everyone piling into lane 1 while lanes 2-10 sit empty.

  9.1 SPREADING OUT READ TRAFFIC

        Incoming Requests
               |
               v
        +---------------+
        | Load Balancer  |
        +--+----+----+---+
           v    v    v
        App1  App2  App3   (each has its own little pool of DB connections)
           |    |    |
           v    v    v
        Replica1 Replica2 Replica3

    Two decisions matter most:
      - Algorithm: "least connections" usually beats plain
        "round robin" for databases, because some queries take way
        longer than others, and you don't want to keep piling MORE
        work onto a replica that's already busy with a slow query.
      - Weighting: if one replica is on weaker hardware, or is
        lagging behind on replication, send it LESS traffic than the
        others.

    Avoid "sticky" routing (always sending the same user to the same
    replica) for general reads — it messes with even distribution —
    unless you specifically need it to solve the "monotonic reads"
    problem from section 5.

  9.2 CONNECTION POOLING

    Opening a brand-new database connection for every single request
    is surprisingly expensive (a bit like re-introducing yourself and
    filling out a form every time you walk into a store you visit
    daily). A connection pool keeps a set of ready-to-use connections
    open, and requests just "borrow" one and "return" it when done.

        WITHOUT A POOL:              WITH A POOL:
        Request                      Request
          |                            |
        Create new connection        Borrow from pool
          |                            |
        Run query                    Run query
          |                            |
        Close connection              Return to pool

    Size the pool carefully: too many connections can overload the
    database; too few, and requests just sit around waiting in line.


--------------------------------------------------------------------------------
  10. HOW IT ALL FITS TOGETHER (the full funnel)
--------------------------------------------------------------------------------

  Here's a real example of how traffic gets reduced, layer by layer,
  starting from 1,000,000 requests:

        1,000,000 requests
              |
              v   Browser Cache filters out ~20%
          800,000
              |
              v   CDN filters out ~60% of what's left
          320,000
              |
              v   Application Cache filters out ~80% of what's left
           64,000
              |
              v   Spread across several Read Replicas
        ~21,000 per replica

    In this made-up example, the DATABASE only ever sees about 6.4% of
    the ORIGINAL traffic. Every layer in front of it did its job of
    absorbing repeat requests so the expensive layer barely had to work.

    That's the whole game: it's not about making the database
    superhuman — it's about making sure it rarely gets bothered at all.


--------------------------------------------------------------------------------
  11. QUICK-REFERENCE DECISION CHECKLIST
--------------------------------------------------------------------------------

  Before you finalize a "high read traffic" design, check:

    [ ] Do I know the read:write ratio for this system? (If it's way
        more reads than writes, this whole toolkit applies.)
    [ ] Are the reads REPEATED (same data asked for over and over)?
        -> If yes, caching is your #1 priority.
    [ ] How stale can this data be? Have I picked a TTL / invalidation
        strategy that actually matches the business need?
    [ ] Is this data public, or private/personalized?
        -> Public: cache aggressively, even at the CDN.
        -> Private: be very careful what you cache and where.
    [ ] What happens on a cache miss, or if the WHOLE cache goes down?
        Will the database survive that sudden hit?
    [ ] Do I have a plan for hot keys / viral content specifically?
        (Request coalescing, early expiration, hot-key replication)
    [ ] Are my users spread across the globe? If so, do I need a CDN
        or regional replicas to cut down physical distance?
    [ ] For anything hitting the database directly — do the important
        queries have the RIGHT indexes?
    [ ] Is there anything expensive to compute that I could precompute
        ahead of time instead of on every single read?
    [ ] Am I actually MEASURING cache hit rate, latency percentiles,
        and database load — or just guessing that it's "probably fine"?


--------------------------------------------------------------------------------
  TL;DR — THE ONE THING TO REMEMBER
--------------------------------------------------------------------------------

  Don't make your expensive, single source-of-truth database answer
  every single question. Put faster, cheaper layers (browser cache,
  CDN, application cache, read replicas) IN FRONT of it, and only let
  requests reach the database when they truly have to — on a cache
  miss, a write, or a request that genuinely needs the freshest
  possible data.

  Every layer you add in front reduces the load on the layer behind it.
  That's the whole strategy, from a fridge sticky-note all the way up
  to a global CDN.

================================================================================
*/

In [ ]:
/*
====================================================================
   HANDLING HIGH READ TRAFFIC - MOCK INTERVIEW STYLE PREP (HINGLISH)
   Format: Interviewer aur Candidate ke beech real conversation
   Goal: System Design interview ke liye ready hona
====================================================================

Kaise use karein yeh notes:
  - Yeh ek REAL interview jaisa conversation hai, top se bottom.
  - Interviewer ke questions padho, khud sochne ki koshish karo,
    phir Candidate ka model answer dekho.
  - Har jagah text diagrams diye hain taaki visually samajh aaye.
  - End mein "RAPID FIRE" section hai jo last-minute revision
    ke liye hai.

Characters:
  INTERVIEWER = Jo sawaal pooch raha hai
  CANDIDATE   = Tum (ideal jawaab de rahe ho)

====================================================================
ROUND 1: PROBLEM SAMAJHNA (Understanding the Read Problem)
====================================================================

INTERVIEWER:
  "Design karo ek system jaisa Twitter ya Instagram feed - jahan
   bahut zyada users content READ karte hain. Kaise handle karoge
   itna zyada READ traffic?"

CANDIDATE:
  "Sir, is problem ka core yeh hai ki zyadatar user-facing apps
   READ-HEAVY hote hain. Matlab, thode log content CREATE karte
   hain (post/video/listing), lekin bahut zyada log usko READ/VIEW
   karte hain. Read-to-write ratio commonly 100:1 hota hai, aur
   viral content ke liye toh usse bhi zyada."

  READ:WRITE RATIO EXAMPLES (yaad rakhne layak):

    System                | Read:Write Ratio | Kyun
    -----------------------|-------------------|---------------------
    Social media feed      | 1000:1            | Bahut viewers, kam posters
    E-commerce catalog     | 10000:1           | Millions browse, kam buy
    News website            | 100000:1          | Content ek baar bane,
                                                   hamesha padha jaye
    Video streaming         | 10000:1           | Upload ek baar, stream
                                                   millions baar
    URL shortener           | 10000:1           | Create ek baar, redirect
                                                   baar baar

  "Toh main pehle problem ko teen dimensions mein todunga: SCALE,
   LATENCY, aur COST. Phir uske baad solution approach batauna."

INTERVIEWER:
  "Scale ka dimension samjhao - numbers ke saath."

CANDIDATE:
  "Maan lijiye 100 Million daily active users hain, aur har user
   din mein 10 baar feed check karta hai:

    100M Users x 10 Views/User = 1 Billion Views/Day
    1 Billion / 86400 sec = ~11,500 Requests/Sec (average)
    Peak traffic mein yeh ~35,000 RPS tak jaa sakta hai
    Agar har response ~50KB hai, toh ~1.75 GB/sec egress bandwidth

   Yeh itna zyada load hai ki EK single database instance is poore
   load ko directly absorb nahi kar sakta - especially jab har
   request multiple queries mein fanout karti hai."

INTERVIEWER:
  "Latency ka dimension?"

CANDIDATE:
  "Yeh doosra bada issue hai. Chahe database itna load handle bhi
   kar le, USER-FACING latency budget bahut tight hota hai:

    Expectation           | Target      | Database Reality
    -----------------------|-------------|-----------------------
    Page Load              | < 200ms     | 500-2000ms (agar direct DB)
    API Response           | < 100ms     | 50-500ms
    Search                 | < 300ms     | 1-5 seconds
    Autocomplete           | < 50ms      | 200-500ms

   Iska matlab hai - agar hum HAR read directly database se serve
   karein, latency budget miss ho jayega. Database query mein
   network hops, query planning, locking, disk I/O - sab time
   leta hai."

INTERVIEWER:
  "Aur cost ka angle?"

CANDIDATE:
  "Database capacity bahut EXPENSIVE hoti hai - CPU, memory,
   storage, replication, backups sab combine hoke cost badhate
   hain. Isके muqable, memory cache ya CDN se repeat reads serve
   karna bahut SASTA hota hai per-request basis pe.

   Toh MAIN GOAL yeh nahi hai ki database har read serve kare.
   MAIN GOAL yeh hai ki REPEATED reads ko cheap, fast layers se
   serve karo, aur database ko sirf CACHE MISSES, WRITES, aur
   FRESH DATA ki zaroorat wali requests ke liye reserve karo."


====================================================================
ROUND 2: CLARIFYING QUESTIONS - Seedha Solution Pe Mat Kudo
====================================================================

INTERVIEWER:
  "Theek hai, ab bataiye kaise design karoge?"

CANDIDATE:
  "Design pe jaane se pehle, main kuch cheezein clarify karna
   chahunga, kyunki yeh decide karengi ki kaunsa tool use karna
   sahi rahega."

CANDIDATE (Clarifying Questions):
  1. "Read:Write ratio kya hai roughly?"
  2. "Reads REPEATED hain (same data baar baar) ya UNIQUE hain
      (har request alag data maangti hai)?"
  3. "Data kitna FRESH hona chahiye? Kya thoda staleness (jaise
      1-5 minute purana data) acceptable hai?"
  4. "Response PUBLIC hai (sabke liye same) ya USER-SPECIFIC/
      PRIVATE hai?"
  5. "Cache miss hone par kya hoga - database survive kar payega?"
  6. "Kya koi HOT OBJECTS honge - jaise viral post ya flash-sale
      product - jo achanak bahut zyada traffic le sakte hain?"
  7. "Users GLOBALLY distributed hain ya ek hi region mein?"

INTERVIEWER:
  "Achha, maan lo: read:write ratio 1000:1 hai, zyadatar reads
   repeated hain (log same feed baar baar dekhte hain), 1-2 minute
   ka staleness chalega, users global hain (Asia, Europe, US sab
   jagah), aur haan, viral posts hote rehte hain."

CANDIDATE:
  "Perfect, is information se clear hai:
     - REPEATED reads hai -> CACHING sabse bada lever hoga
     - Staleness tolerate hoti hai -> TTL-based approach kaam
       karega
     - Global users -> CDN/Edge zaroori hai
     - Viral posts -> HOT KEY aur STAMPEDE protection chahiye

   Ab main READ SCALING TOOLKIT explain karta hoon - yeh saare
   tools hain jo hum combine karke use karenge."


====================================================================
ROUND 3: READ SCALING TOOLKIT - Overview
====================================================================

INTERVIEWER:
  "Toolkit batao - kya kya options hote hain?"

CANDIDATE:
  "Sir, 6 main techniques hoti hain, aur practically hum sabko
   COMBINE karke use karte hain:

    Technique              | Kya Karta Hai                    | Impact
    ------------------------|-----------------------------------|------------------
    Caching                 | Repeated reads offload karta hai  | Bahut bada DB load reduction
    Read Replicas            | DB read capacity badhata hai      | Zyada capacity, but lag
    CDN / Edge                | Global low latency deta hai       | Kam latency, kam origin load
    Database Optimization     | Queries fast banata hai           | Faster remaining reads
    Pre-computation            | Pehle se compute karta hai         | Fast reads expensive data ke liye
    Load Balancing              | Traffic distribute karta hai      | Hotspots avoid karta hai

   MERA DEFAULT ORDER hoga:
     1. Repeated reads ke liye CACHE lagao
     2. Global cacheable content ke liye CDN use karo
     3. Remaining DB reads ke liye READ REPLICAS add karo
     4. Slow queries ko OPTIMIZE karo (indexes)
     5. Expensive derived views ko PRE-COMPUTE karo

   Main ab har technique ko detail mein explain karta hoon,
   sabse impactful (caching) se shuru karke."


====================================================================
ROUND 4: CACHING - The First Line of Defense
====================================================================

INTERVIEWER:
  "Caching ka basic idea samjhao."

CANDIDATE:
  "Idea simple hai: agar same data baar baar padha ja raha hai,
   toh uski EK COPY faster storage mein rakh do - taaki har baar
   database tak na jaana pade.

   Design questions yeh hain: KYA cache karein, KAHAN cache karein,
   KITNI DER tak stale rehne dein, aur SOURCE change hone pe kaise
   INVALIDATE karein."

INTERVIEWER:
  "Caching HIERARCHY explain karo - kitne layers hote hain?"

CANDIDATE:
  "Modern systems mein MULTIPLE layers hote hain caching ke,
   har ek alag access pattern ke liye optimized:

    DIAGRAM:

      [User Request]
             |
             v
      [Browser Cache] --local lookup, 0ms--
             |  (miss)
             v
      [CDN Cache] --10-50ms latency--
             |  (miss)
             v
      [Load Balancer]
             |
             v
      [App Server]
             |
             v
      [Application Cache - Redis/Memcached] --1-5ms latency--
             |  (miss)
             v
      [DB Buffer Pool] --0.1-1ms latency--
             |  (miss)
             v
      [Database - Disk Storage]

   Har layer, agli layer par pahunchne se PEHLE request ko
   intercept kar leta hai. Goal yeh hai ki MAXIMUM requests
   FASTEST layers se serve ho jayein.

    Layer               | Latency    | Capacity          | Best For
    ---------------------|------------|--------------------|------------------
    Browser cache         | 0ms        | Limited per user    | Static assets
    CDN cache              | 10-50ms    | Massive             | Static/regional content
    Application cache       | 1-5ms      | Medium (memory)     | Query results, sessions
    Database cache            | 0.1-1ms    | Small (RAM)         | Query plans, hot pages"

INTERVIEWER:
  "Application-level caching kaise implement karoge? Pattern
   batao."

CANDIDATE:
  "CACHE-ASIDE pattern (lazy loading) sabse common hai:

    DIAGRAM (Cache Hit path):

      Client --GET /user/123--> [App Server]
                                      |
                              GET user:123 (Redis)
                                      |
                              [Cache Hit! User Data mili]
                                      |
                                      v
                       Return cached data (~2ms fast)

    DIAGRAM (Cache Miss path):

      Client --GET /user/123--> [App Server]
                                      |
                              GET user:123 (Redis)
                                      |
                              [Cache Miss - null aaya]
                                      |
                              SELECT * FROM users WHERE id=123
                                      |
                              [Database se data mila (~50ms)]
                                      |
                              SET user:123 (with TTL) -- next time ke liye cache
                                      |
                                      v
                                Return data

   Har READ pe, pehle CACHE check karo. Agar mil gaya, return kar
   do. Agar nahi mila, DATABASE se lao aur CACHE mein store kar do
   future reads ke liye."

INTERVIEWER:
  "Iska impact kitna hota hai numbers mein?"

CANDIDATE:
  "Agar CACHE HIT RATE 90% hai, toh us endpoint ke liye DATABASE
   READ TRAFFIC roughly 10x KAM ho jata hai. Interview mein
   IMPORTANT METRIC hai CACHE HIT RATE, sirf 'cache lagaya hai'
   bolna kaafi nahi hai - hit rate batana zaroori hai."


====================================================================
ROUND 5: CACHE INVALIDATION STRATEGIES
====================================================================

INTERVIEWER:
  "Caching ka hardest part kya hai?"

CANDIDATE:
  "INVALIDATION sir. Cached data source-of-truth ki ek COPY hai.
   Jab source change hota hai, cache STALE ho sakta hai. Strategy
   business ki STALENESS TOLERANCE ke according choose karni
   chahiye."

INTERVIEWER:
  "Pehli strategy batao - TTL."

CANDIDATE:
  "TIME-TO-LIVE (TTL) sabse SIMPLE approach hai - cached data
   FIXED TIME ke baad automatically expire ho jata hai.

    DIAGRAM:

      [Write to DB] --> [Data in Cache] --> [TTL Countdown]
             |
             v
      [Cache Expires] --> [Next Read = MISS] --> [Refresh from DB]

   5-minute TTL matlab data 5 minute tak PURANA ho sakta hai.
   News feed/product catalog ke liye theek hai, lekin BANK BALANCE
   ke liye NAHI chalega."

INTERVIEWER:
  "Doosri strategy - Write-Through?"

CANDIDATE:
  "WRITE-THROUGH mein hum DATABASE aur CACHE dono ko SYNCHRONOUSLY
   update karte hain WRITE ke waqt hi.

    DIAGRAM:

      Client --UPDATE user 123--> [App Server]
                                        |
                              UPDATE users SET ... (Database)
                                        |
                                   [Success]
                                        |
                              SET user:123 (new data) (Cache)
                                        |
                                   [OK]
                                        |
                                        v
                                 Success return to client

   Yeh cache ko FRESH rakhta hai, lekin WRITE LATENCY badh jaata
   hai - aur EDGE CASE handle karni padti hai: agar DB update ho
   gaya lekin CACHE update fail ho gaya, toh kya karein?"

INTERVIEWER:
  "Teesri - Event-Based Invalidation?"

CANDIDATE:
  "Yeh COMPLEX systems, especially MICROSERVICES ke liye acha hai:

    DIAGRAM:

      [Write to DB] --> [Emit Event] --> [Message Queue - Kafka/RabbitMQ]
                                                |
                        ---------------------------------------
                        |                   |                 |
                        v                   v                 v
                Cache Invalidator    Cache Invalidator   Cache Invalidator
                   Service A            Service B           Service C
                        |                   |                 |
                Delete Cache Key    Delete Cache Key   Delete Cache Key

   Jab data change hota hai, hum EVENT publish karte hain.
   Subscribers event receive karke apna LOCAL CACHE invalidate
   karte hain. Yeh WRITER ko CACHE se DECOUPLE kar deta hai -
   naye services/caches add karna easy ho jata hai."

INTERVIEWER:
  "In teeno mein se konsa choose karoge, aur kab?"

CANDIDATE:
  "Comparison Table:

    Pattern           | Consistency | Write Speed        | Complexity | Best For
    --------------------|--------------|----------------------|-------------|--------------------------
    Cache-aside + TTL     | Eventual     | Low overhead          | Low         | Most apps, feeds, catalogs
    Write-through           | Strong per key| Higher overhead        | Medium      | Small critical read models
    Event-based               | Eventual     | Low overhead           | High        | Microservices, multi-cache

   Main ZYADATAR applications ke liye CACHE-ASIDE + TTL se
   SHURU karunga, kyunki simple hai. Agar business FRESHER data
   demand kare, tab EVENT-BASED ya WRITE-THROUGH move karunga."

INTERVIEWER:
  "Kya cache karna chahiye, aur kya NAHI?"

CANDIDATE:
  "Good Cache Candidates:
     - User profiles
     - Product details
     - News feed content
     - Search results
     - Configuration data
     - Session data

   Poor Cache Candidates:
     - Rapidly changing prices (jinhe latest quote chahiye)
     - Bank account balances
     - One-time auth tokens
     - Highly personalized data
     - Large binary blobs (CDN use karo instead)
     - Bahut low read-rate wala data

   MAIN QUESTION yeh hai: 'Kya caching itna FAYDA de rahi hai ki
   FRESHNESS maintain karne ki COMPLEXITY justify ho jaye?'
   Jo data FREQUENTLY READ hota hai aur SLOWLY CHANGE hota hai,
   woh best candidate hai."


====================================================================
ROUND 6: HOT KEYS AUR CACHE STAMPEDE - Interviewer Yahan Depth Test Karega
====================================================================

INTERVIEWER:
  "Achha, ek viral tweet ka scenario lo. Millions of reads ek
   single cache key pe aa rahe hain. Kya problem ho sakti hai?"

CANDIDATE:
  "Yeh do RELATED problems create karta hai - HOT KEY aur CACHE
   STAMPEDE - dono common interview follow-ups hain.

   HOT-KEY PROBLEM: Sharded cache mein har KEY ek hi NODE pe rehta
   hai. Agar ek key ko traffic ka bada SHARE mil raha hai, toh
   SIRF EK node bottleneck ban jata hai, baaki cluster IDLE rehta
   hai. Sharding badhane se yeh problem SOLVE nahi hoti, kyunki
   traffic EK key pe hai, spread nahi hai many keys mein.

   CACHE STAMPEDE (thundering herd / dogpile effect): Jab hot key
   EXPIRE hoti hai, saare IN-FLIGHT requests EK SAATH MISS karte
   hain, aur SAB ek saath DATABASE se same value RECOMPUTE karne
   ki koshish karte hain.

    DIAGRAM:

      [Hot key just expired]
             |
             v
      [Sudden surge of identical requests]
             |
             v
      GET hot:key --> MISS (all at once)
             |
             v
      [Every request recomputes SAME value from DB]
             |
             v
      [Database - nearly idle se OVERLOADED in a fraction of second]

   Database jo pehle ISI KEY ke liye almost IDLE tha (kyunki cache
   sab handle kar raha tha), achanak ek hi second mein OVERLOAD ho
   jata hai."

INTERVIEWER:
  "Kaise solve karoge in dono problems ko?"

CANDIDATE:
  "5 TECHNIQUES hain, jo COMBINE hoke kaam karti hain:

    Technique                   | Kaam                                            | Solve Karta Hai
    ------------------------------|--------------------------------------------------|------------------
    Request Coalescing             | Miss pe SIRF PEHLI request recompute karti hai;   | Stampede
    (single-flight)                  baaki concurrent requests uska RESULT WAIT karti hain
    Locking + stale fallback        | Pehli miss short lock leti hai refresh ke liye;    | Stampede
                                       baaki thodi STALE value serve karti hain jab tak ready na ho
    Probabilistic early expiration    | Requests randomly TTL se pehle hi refresh karti hain, | Stampede
                                        isliye expiry TIME MEIN SPREAD ho jati hai
    Hot-key replication               | Popular key ko MULTIPLE nodes pe store karo         | Hot Key
                                       (ya local in-process cache mein replicate karo)
    Local (in-process) caching        | Hottest objects ko app-server memory mein          | Hot Key
                                       kuch seconds ke liye cache karo, shared cache ke aage

   MAIN PEHLE 'REQUEST COALESCING' bataunga - yeh USUALLY PEHLA
   ANSWER hota hai. Iska matlab: 1000 concurrent misses ko database
   pe alag alag hit karne ki jagah, EK request kaam kare aur baaki
   uska RESULT WAIT karein. '1000 misses' -> 'SINGLE database
   query' ban jata hai.

   PROBABILISTIC EARLY EXPIRATION synchronized-expiry problem ko
   solve karta hai - saari copies EK saath expire hone ki jagah,
   TTL ke close aate hi RANDOM chance se refresh hoti rehti hain,
   isliye key kabhi bhi 'sabke liye ek saath cold' nahi hoti.

   HOT-KEY IMBALANCE ke liye, REPLICATION lever hai - key ko
   MULTIPLE SUB-KEYS mein baant ke different nodes pe store karo,
   ya HAR APP SERVER ke local memory mein SHORT-LIVED copy rakho."

INTERVIEWER:
  "Do aur cases batao jo log bhool jaate hain - negative caching
   aur cache failure."

CANDIDATE:
  "NEGATIVE CACHING: Jo keys DATABASE mein EXIST hi NAHI karte
   (deleted/not-found), unka 'NOT FOUND' result bhi THODI DER ke
   liye CACHE karo. Warna, MISSING/DELETED object ke liye BAAR
   BAAR requests HAR BAAR cache miss karengi aur DATABASE tak
   jayengi - yeh CHEAP ATTACK VECTOR bhi ban sakta hai.

   CACHE FAILURE FALLBACK: Agar CACHE khud DOWN ho jaye ya COLD
   RESTART ho, toh kya hoga? Agar HAR request seedha DATABASE tak
   fall through kare, database ko FULL READ LOAD milega jiske liye
   woh SIZED hi nahi tha. REQUEST COALESCING, SHORT LOCAL CACHES,
   aur LOAD SHEDDING - yeh sab database ko COLD/FAILED cache
   survive karne mein HELP karte hain.

   INTERVIEW TIP: Hot keys aur stampede protection UNPROMPTED bhi
   RAISE karo - yeh common FAILURE POINT hai otherwise reasonable
   caching designs mein."


====================================================================
ROUND 7: READ REPLICAS - Scaling Database Reads
====================================================================

INTERVIEWER:
  "Cache miss hone ke baad bhi bahut zyada reads DB pe aa rahi
   hain. Ab kya karoge?"

CANDIDATE:
  "Ab READ REPLICAS add karunga. Idea simple hai: EK PRIMARY
   database saare WRITES handle karta hai, aur uske CHANGES ek ya
   zyada REPLICAS pe REPLICATE hote hain. READ-ONLY queries in
   replicas mein DISTRIBUTE ki jaati hain.

    DIAGRAM (Primary-Replica Architecture):

      [App Server 1]   [App Server 2]   [App Server N]
             \                |                /
              \               |               /
               v              v              v
            [Proxy / Router - Splits Read/Write]
               /              |              \
          WRITES          READS            READS
              |               |                |
              v               v                v
         [Primary]      [Replica 1]      [Replica 2]
         Handles Writes  Handles Reads   Handles Reads
              |------ Replication Stream ------->|
              |------ Replication Stream -------------->|

   Yeh READ CAPACITY badhata hai, lekin PERFECTLY LINEAR nahi hai
   - replicas REPLICATION BANDWIDTH, STORAGE COST, CONNECTION
   LIMITS share karte hain."

INTERVIEWER:
  "Important warning - kya replicas HAR problem solve kar denge?
   Jaise hot row ka?"

CANDIDATE:
  "NAHI sir! Yeh SIRF tab help karte hain jab LOAD MANY ROWS mein
   SPREAD ho. Agar EK SINGLE HOT ROW/KEY hai (jaise viral post),
   toh WOHI CONTENTION HAR REPLICA pe bhi hogi - kyunki sab
   replicas mein WOHI row hai. Isliye VIRAL OBJECT ko CACHING aur
   HOT-KEY TECHNIQUES chahiye, MORE REPLICAS se woh solve nahi
   hoga."

INTERVIEWER:
  "Replication ke do methods batao - Async aur Sync."

CANDIDATE:
  "ASYNCHRONOUS REPLICATION:

    DIAGRAM:

      Client --INSERT INTO orders...--> [Primary DB]
                                              |
                                        Write to disk
                                              |
                                     ACK - Commit successful
                                              |
                                              v
                                    Client continues (turant)
                                              |
                              (10ms-1sec baad) Replicate changes
                                              |
                                              v
                                     [Replica] - Apply changes
                                              |
                                     Replica ab CONSISTENT hai

   PRIMARY REPLICA ka WAIT nahi karta - WRITE turant ACK ho jata
   hai. Fast hai, lekin REPLICATION LAG create karta hai - matlab
   replica se READ karne pe STALE data mil sakta hai.

    Pros                                | Cons
    --------------------------------------|--------------------------------
    Primary replica ka wait nahi karta     | Replication lag (stale reads)
    Kam commit latency                       | Data loss agar primary fail ho
    Read replicas traffic absorb karte hain    | Read-your-writes inconsistency

   SYNCHRONOUS REPLICATION:

    DIAGRAM:

      Client --INSERT...--> [Primary DB] --Write to disk-->
                                    |
                             Replicate changes
                                    |
                                    v
                              [Replica] --Persist to WAL--> ACK
                                    |
                                    v
                          [Primary] --ACK - Commit successful-->
                                    |
                                    v
                             Client continues

   PRIMARY tab tak ACK NAHI karta jab tak KAM SE KAM EK REPLICA
   confirm na kare ki change uski WAL (write-ahead log) mein
   DURABLY receive ho gaya hai (apply hona zaroori nahi, isliye
   thoda REPLAY LAG phir bhi reh sakta hai).

    Pros                                 | Cons
    ---------------------------------------|--------------------------------
    Replica confirm karta hai commit se pehle| Zyada write latency
    Stronger durability                        | Replica fail hone pe write block
    Kam failover data-loss risk                  | Zyada complex coordination

   PRACTICALLY, ZYADATAR PRODUCTION SYSTEMS ASYNC use karte hain
   aur APPLICATION ko BRIEF INCONSISTENCY tolerate karne layak
   design karte hain. SYNC sirf critical data ke liye reserve
   hota hai."

INTERVIEWER:
  "Replication lag se user ko apna hi update purana dikh sakta
   hai. Kaise solve karoge?"

CANDIDATE:
  "Yeh classic scenario hai - USER PROFILE UPDATE karta hai,
   REFRESH karta hai, PURANA data dikhta hai (kyunki WRITE
   primary pe gaya, lekin READ LAGGING REPLICA se aayi).

   3 PATTERNS solve karte hain:

    1. READ-YOUR-WRITES CONSISTENCY:
       Track karo ki har user ne LAST kab WRITE kiya. Thodi der
       ke liye (jaise 5 sec) uske READS ko PRIMARY pe route karo.

    2. MONOTONIC READS:
       STICKY SESSIONS ya REPLICA TOKEN use karo, taaki user
       HAMESHA SAME ya NEWER replica pe rahe. Isse data 'PEECHE'
       jaate hue nahi dikhega alag-alag lag wale replicas ke beech
       move karne se.

    3. LAG MONITORING:
       Har REPLICA ka REPLICATION LAG track karo. Sirf un
       REPLICAS ko reads bhejo jo ENDPOINT ke FRESHNESS BUDGET
       ke andar hain. Agar replica bahut zyada peeche hai, use
       ROTATION se HATA do jab tak catch up na kare.

    Solution              | Kaam                       | Trade-off
    -------------------------|-------------------------------|----------------------------
    Read-your-writes           | Recent writers ko primary   | Primary pe zyada load
                                  pe route karo
    Monotonic reads              | Sticky session same replica    | Uneven distribution
                                    pe rakho
    Lag monitoring                 | Sirf low-lag replicas use     | Kam available replicas
                                     karo"

INTERVIEWER:
  "Reads ko replicas mein route kaise karoge? Options batao."

CANDIDATE:
  "Do main approaches hain:

    1. APPLICATION-LEVEL ROUTING: Application EXPLICITLY decide
       karta hai HAR query kahan bhejni hai. FINE-GRAINED control
       milta hai, lekin app code mein CHANGES chahiye.

    2. PROXY-BASED ROUTING: Ek DATABASE PROXY/DRIVER layer app
       aur DB ke beech baithta hai, aur QUERY TYPE (read/write)
       ke basis pe automatically route karta hai.

    DIAGRAM:

      [App Servers]
             |
       INSERT/UPDATE/DELETE -----> [Primary]
             |
          SELECT ---> [Database Proxy - ProxySQL/Pgpool-II]
                              |
                        ------------------
                        |                |
                        v                v
                  [Replica 1]      [Replica 2]

   ProxySQL MySQL ke liye common hai; PostgreSQL setups mein
   APPLICATION ROUTING ya PGPOOL-II common hai. PGBOUNCER SIRF
   CONNECTION POOLER hai, GENERAL ROUTER NAHI."


====================================================================
ROUND 8: CDN - Serving Content at the Edge
====================================================================

INTERVIEWER:
  "Ab bataiye, agar user Tokyo mein hai aur server San Francisco
   mein hai, toh latency ka kya scene hoga?"

CANDIDATE:
  "Yeh PHYSICAL DISTANCE ka problem hai. Request THOUSANDS OF
   MILES travel karti hai, aur network round-trips hi NOTICEABLE
   latency add kar dete hain - APPLICATION PROCESSING SHURU hone
   se PEHLE hi.

   Solution hai CDN (Content Delivery Network) - yeh content ko
   WORLDWIDE EDGE LOCATIONS pe CACHE karta hai. User ORIGIN
   server tak jaane ki jagah NEAREST EDGE se serve hota hai."

INTERVIEWER:
  "CDN kaise kaam karta hai, diagram banao."

CANDIDATE:
  "DIAGRAM:

    [User in Tokyo] ---> [Edge Server Tokyo ~20ms] --Cache Hit-->
                                                       Fast Response!

    [User in London] ---> [Edge Server London ~20ms] --Cache Hit-->
                                                        Fast Response!

    [User in New York] ---> [Edge Server NY ~20ms] --Cache Hit-->
                                                      Fast Response!

    Agar KISI edge pe CACHE MISS ho jaye:
      Tokyo Edge --Cache Miss: 150ms--> [Origin Server San Francisco]
      London Edge --Cache Miss: 200ms--> [Origin Server San Francisco]
      NY Edge --Cache Miss: 80ms--> [Origin Server San Francisco]

   JAB request CACHED entry se MATCH karti hai NEAREST edge pe,
   woh ORIGIN tak POHUNCHTI HI NAHI. IMPROVEMENT SUBSTANTIAL ho
   sakta hai - exact savings GEOGRAPHY, HIT RATE, connection reuse
   pe depend karta hai."

INTERVIEWER:
  "CDN pe kya kya PUT karoge?"

CANDIDATE:
  "Content Type              | TTL          | Cache-Control Header
   -----------------------------|---------------|--------------------------------
   Static images                 | 1 saal        | max-age=31536000, immutable
   CSS/JS (versioned)              | 1 saal        | max-age=31536000, immutable
   HTML pages                        | 5-60 min      | max-age=300, stale-while-revalidate=60
   API responses                       | 1-5 min       | max-age=60, s-maxage=300
   User-specific data                    | Cache mat karo| private, no-store

   STATIC ASSETS ke liye main CONTENT ADDRESSING use karunga -
   filename mein content ka HASH include karo (jaise
   app.abc123.js). Content change hone pe FILENAME bhi change ho
   jata hai, isliye users AUTOMATICALLY naya version pate hain,
   aur purani file HAMESHA cache ho sakti hai (kabhi stale nahi
   hoti kyunki naam hi badal gaya)."

INTERVIEWER:
  "API response caching CDN pe kaise karoge?"

CANDIDATE:
  "s-maxage HEADER use karunga - yeh SHARED CACHES (jaise CDN)
   ko RESPONSE ko BROWSER se ZYADA DER tak HOLD karne deta hai.

   ISKE SAATH stale-while-revalidate PAIR karunga - yeh CACHE ko
   batata hai ki EXPIRY ke TURANT BAAD bhi EXISTING copy SERVE
   karo, aur BACKGROUND mein REFRESH karte raho.

   Result: USER ko FAST RESPONSE milta hai, kabhi ORIGIN pe WAIT
   nahi karta. SIRF COST yeh hai ki EXPIRY ke BAAD KI PEHLI
   request THODI STALE data dekh sakti hai - lekin yeh BADLY
   OUTDATED data se BEHTAR hai."

INTERVIEWER:
  "Edge computing kya hota hai?"

CANDIDATE:
  "Modern CDNs EDGE PE CODE RUN kar sakte hain (Cloudflare Workers,
   Lambda@Edge) - LIGHTWEIGHT routing, authentication checks,
   localization, A/B tests, LIMITED personalization ke liye,
   BINA HAMESHA origin tak jaane ke.

    DIAGRAM:

      [User Request] --> [Edge Server] --> [Edge Function]
                                                  |
                                          Cached? 
                                          /          \
                                       YES              NO
                                        |                |
                                        v                v
                              Return Cached +      Fetch from Origin
                              Personalization             |
                                                    Cache at Edge

   WARNING: PERSONALIZATION CACHEABILITY KAM kar sakti hai agar
   HAR USER ko UNIQUE response mile. Isliye PUBLIC CACHEABLE data
   ko CHHOTE USER-SPECIFIC OVERLAYS se SEPARATE rakhna chahiye."

INTERVIEWER:
  "CDN cache invalidate kaise karoge?"

CANDIDATE:
  "4 METHODS:
     1. TTL-BASED EXPIRATION: Content naturally expire ho jaye.
        Simple lekin COARSE-GRAINED.
     2. PURGE BY URL: Specific URLs ko EXPLICITLY invalidate karo
        jab woh change hon.
     3. PURGE BY TAG: Content ko TAG karke cache karo, phir TAG se
        invalidate karo (multiple related URLs ek saath).
     4. VERSIONED URLs: STATIC ASSETS ke liye SABSE CLEAN approach
        - content change hone pe URL bhi change karo, purani
        cached asset naturally EXPIRE ho jayegi."


====================================================================
ROUND 9: DATABASE OPTIMIZATION
====================================================================

INTERVIEWER:
  "Caching, CDN, replicas sab hone ke baad bhi kuch requests DB
   tak pohunchengi. Unko fast kaise banaoge?"

CANDIDATE:
  "SABSE PEHLE INDEXING. Indexes ek BROAD SCAN ko TARGETED LOOKUP
   mein badal dete hain.

    DIAGRAM (Without Index):
      SELECT * FROM orders WHERE customer_id = 123
              |
              v
        Scan 10M rows --> 5 seconds (SLOW!)

    DIAGRAM (With Index):
      Same Query
              |
              v
        Index Lookup --> Find 150 rows --> 5ms (FAST!)

   customer_id pe INDEX lagane se FULL TABLE SCAN, EK FAST LOOKUP
   ban jata hai. IMPROVEMENT depend karta hai SELECTIVITY, TABLE
   SIZE, aur QUERY SHAPE pe.

   COMPOSITE INDEXES: Multiple columns pe filter karne wali
   queries ke liye, COMPOSITE INDEX use karo. INDEX ORDER, QUERY
   PATTERN ko MATCH karna chahiye - PEHLE FILTER columns, PHIR
   SORT columns."

INTERVIEWER:
  "Denormalization kya hai, aur kab use karoge?"

CANDIDATE:
  "NORMALIZED schema REDUNDANCY minimize karta hai, lekin data
   reassemble karne ke liye JOINS chahiye hote hain - jo SCALE pe
   EXPENSIVE hain.

   DENORMALIZATION storage ko TRADE karta hai READ PERFORMANCE ke
   liye - data ko PRE-JOIN karke rakhte hain.

    DIAGRAM:

      NORMALIZED (Join-Heavy):
        [Orders] --3-Way JOIN--> [Customers] + [Products]

      DENORMALIZED (Read-Optimized):
        [Orders_Denormalized] 
          - customer_name (already included)
          - customer_email (already included)
          - product_name (already included)
          - product_price (already included)
        --> SINGLE TABLE READ (fast!)

    Aspect       | Normalized              | Denormalized
    ---------------|----------------------------|--------------------------
    Read path       | Join-heavy                | Simpler for common views
    Write path        | Ek canonical copy          | Derived copies update karni padti hain
    Storage             | Kam redundant                 | Zyada redundant
    Use case               | OLTP, write-heavy                | Analytics, read-heavy

   MAIN DENORMALIZATION USE KARUNGA jab READ PERFORMANCE CRITICAL
   ho aur REDUNDANT DATA maintain karne ki COMPLEXITY afford ho."

INTERVIEWER:
  "Slow query ko debug kaise karoge?"

CANDIDATE:
  "EXPLAIN ANALYZE use karunga yeh samajhne ke liye ki QUERY
   PLANNER kya kar raha hai.

    Problem                    | Solution
    ------------------------------|--------------------------------
    Full table scan                | Appropriate index add karo
    Bahut zyada JOINs                | Denormalize ya materialized view
    Bada result set                     | LIMIT add karo, pagination karo
    N+1 queries                           | JOIN ya batch fetching use karo
    Expensive aggregations                  | Materialized view mein pre-compute karo"


====================================================================
ROUND 10: PRE-COMPUTATION AUR MATERIALIZED VIEWS
====================================================================

INTERVIEWER:
  "Kuch queries indexing ke baad bhi slow rehti hain - jaise
   millions rows aggregate karna. Kya karoge?"

CANDIDATE:
  "In cases mein result ko PEHLE SE COMPUTE karo - PRE-COMPUTATION.

   MATERIALIZED VIEW query ka RESULT ek PHYSICAL TABLE ki tarah
   STORE kar leta hai. Har request pe COMPUTE karne ki jagah, EK
   BAAR compute karo, PHIR PRE-COMPUTED result read karo.

   TRADE-OFF: FRESHNESS ke badle SPEED milti hai. View SIRF
   'LAST REFRESH' tak hi CURRENT hai, jab tak INCREMENTAL REFRESH
   ya EVENT-DRIVEN UPDATE na ho.

   IMPORTANT DETAIL: Plain 'REFRESH MATERIALIZED VIEW' EXCLUSIVE
   LOCK leta hai aur REBUILD ke dauran READS ko BLOCK kar deta hai
   - yeh READ-HEAVY system ke GOAL ke KHILAF jaata hai! Isliye
   'REFRESH MATERIALIZED VIEW CONCURRENTLY' use karo - yeh
   READERS ko block NAHI karta, lekin REFRESH SLOW hoti hai aur
   EK UNIQUE INDEX chahiye hoti hai."

INTERVIEWER:
  "Alag alag PRE-COMPUTED data stores kaunse use karoge?"

CANDIDATE:
  "DIAGRAM (Serving Architecture):

    [Primary Database]
           |
           v
    [Batch Job - Hourly/Daily]
           |
           v
    [Compute & Transform]
           |
    -------------------------------
    |            |                |
    v            v                v
  [Redis]   [Elasticsearch]  [ClickHouse]
  Key-value   Search Queries   Analytics
  lookups
    \            |               /
     \           |              /
      -----------------------------
                  |
                  v
           [App Servers - Serving Layer]

   REDIS: key-value lookups aur LEADERBOARDS ke liye
   ELASTICSEARCH: FULL-TEXT SEARCH ke liye
   CLICKHOUSE/DRUID: ANALYTICAL QUERIES aur EVENT DATA ke liye"

INTERVIEWER:
  "Social feeds ke liye pre-computation kaise apply hoti hai?"

CANDIDATE:
  "Yeh FANOUT-ON-WRITE hai. Computation ko READ TIME se WRITE
   TIME pe SHIFT karte hain:

    DIAGRAM:

      [User Posts Content]
             |
             v
      [Background Worker]
             |
      -----------------------------
      |            |               |
      v            v               v
  Generate Feed  Generate Feed  Generate Feed
  Follower 1     Follower 2     Follower N
      |            |               |
      -----------------------------
                    |
                    v
           [Redis - Pre-generated Feeds]

      LATER: [User Opens App] --> Read Pre-generated Feed
                                --> Cached Feed Response (FAST!)

   Post-time pe COMPUTE karne se, APP-OPEN time pe SIRF EK FAST
   CACHE LOOKUP karna padta hai. HIGH-FOLLOWER authors ke liye
   HYBRID approach chahiye - unke posts ke liye FANOUT-ON-READ,
   normal authors ke liye FANOUT-ON-WRITE (jaisa humne pichle
   topic mein detail mein discuss kiya tha)."


====================================================================
ROUND 11: LOAD BALANCING AUR CONNECTION POOLING
====================================================================

INTERVIEWER:
  "Multiple app servers aur replicas ke beech traffic kaise
   spread karoge?"

CANDIDATE:
  "DIAGRAM:

      [Incoming Requests]
             |
             v
      [Load Balancer]
             |
    -----------------------------
    |             |              |
    v             v              v
  [App Server1] [App Server2] [App Server N]
  Connection    Connection    Connection
  Pool          Pool          Pool
    |             |              |
    -----------------------------
                   |
       ------------------------
       |            |          |
       v            v          v
   [Replica1]  [Replica2]  [Replica3]

   Har APP SERVER apna OWN connection pool maintain karta hai,
   isliye LOAD BALANCER, HTTP requests aur DATABASE connection
   load - DONO ko EK SAATH distribute karta hai."

INTERVIEWER:
  "Balancing algorithm kaunsa use karoge - round-robin ya
   least-connections?"

CANDIDATE:
  "LEAST-CONNECTIONS USUALLY BEHTAR hota hai DATABASE READS ke
   liye, kyunki QUERY DURATIONS VARY karti hain - hum nahi chahte
   ki EK ALREADY BUSY replica (jo ek SLOW SCAN kar rahi hai) pe
   AUR queries daal di jayein.

   ROUND-ROBIN theek hai jab REPLICAS IDENTICAL hon aur REQUESTS
   UNIFORM hon.

   WEIGHTING bhi important hai - agar EK replica DIFFERENT
   HARDWARE pe hai ya REPLICATION LAG catch up kar raha hai, toh
   WEIGHTED ROUTING use karo, isko KAM traffic bhejo.

   IP-HASH jaisi STICKY strategies AVOID karunga replica reads ke
   liye, kyunki EK CLIENT ko EK REPLICA pe PIN karna EVEN
   DISTRIBUTION ko UNDERMINE karta hai - JAB TAK MONOTONIC READS
   ke liye SPECIFICALLY zaroori na ho."

INTERVIEWER:
  "Connection pooling kyun zaroori hai?"

CANDIDATE:
  "DATABASE CONNECTIONS banana EXPENSIVE hai. Har request ke
   liye NAYA connection kholna AVOIDABLE OVERHEAD add karta hai.

    DIAGRAM (Without Pool):
      Request --> Create Connection --> Query --> Close Connection
      (Har baar SLOW - connection create/close overhead)

    DIAGRAM (With Pool):
      Request --> Get from Pool --> Query --> Return to Pool
      (FAST - connection REUSE hota hai)

   CONNECTION POOL ek BOUNDED SET of REUSABLE connections
   maintain karta hai. POOL SIZE CAREFULLY choose karni chahiye -
   ZYADA connections DATABASE ko OVERLOAD kar sakte hain, KAM
   connections REQUESTS ko QUEUE karwa denge."


====================================================================
ROUND 12: PUTTING IT ALL TOGETHER - Complete Architecture
====================================================================

INTERVIEWER:
  "Ab final architecture batao, jaha sab kuch combine ho."

CANDIDATE:
  "COMPLETE FLOW yeh hoga:

    LAYER 1 - CLIENT LAYER:
      [Users] --> [Browser Cache] (static assets, kuch APIs)

    LAYER 2 - EDGE LAYER:
      Browser --Cache Miss--> [CDN Edge Servers Worldwide]

    LAYER 3 - APPLICATION LAYER:
      CDN --Cache Miss--> [Load Balancer] --> [App Server Cluster]
                                                     |
                                              [Redis Cache Cluster]
                                              (Query Results, Session Data)

    LAYER 4 - DATA LAYER:
      Cache --Cache Miss--> [Primary] --Async Replication-->
                                              [Replica 1]
                                              [Replica 2]
                                              [Replica 3]
                              +
                         [Materialized Views - Pre-computed Data]

   REQUEST FLOW:
     1. BROWSER CACHE check pehle
     2. CDN static assets aur cached PUBLIC API responses serve
        karta hai
     3. LOAD BALANCER available app server pe route karta hai
     4. APPLICATION CACHE (Redis) check hoti hai
     5. Miss hone pe READ REPLICA se query hoti hai (agar
        stale-tolerant read allow hai)
     6. Response wapas UPAR ki layers se hoke jaata hai"

INTERVIEWER:
  "Har layer kitna traffic REDUCE karti hai, numbers batao."

CANDIDATE:
  "TRAFFIC REDUCTION FUNNEL EXAMPLE:

      1,000,000 Requests aaye
             |
      [Browser Cache] -20% reduction
             |
      800,000 requests aage badhi
             |
      [CDN] -60% reduction
             |
      320,000 requests aage badhi
             |
      [App Cache] -80% reduction
             |
      64,000 requests aage badhi
             |
      [Read Replicas] - distributed across replicas
             |
      ~21,000 requests PER REPLICA

   IS EXAMPLE mein, DATABASE ko SIRF 6.4% ORIGINAL TRAFFIC dikhta
   hai! Exact number HIT RATES aur USER BEHAVIOR pe depend karta
   hai, lekin PATTERN yehi hai - HAR LAYER agli layer ka LOAD
   REDUCE karti hai."


====================================================================
ROUND 13: CLOSING - Final Summary
====================================================================

INTERVIEWER:
  "Theek hai, poore design ka summary do."

CANDIDATE:
  "Sure sir, meri final design yeh hai:

   1. Pehle READ:WRITE RATIO ESTIMATE karunga aur pattern
      (repeated vs unique reads, staleness tolerance) samjhunga.

   2. CACHING sabse bada LEVER hai - CACHE-ASIDE + TTL se shuru
      karunga, HIT RATE monitor karunga.

   3. Multi-layer CACHING HIERARCHY use karunga - Browser -> CDN
      -> Application Cache -> DB Buffer Pool.

   4. HOT KEYS aur CACHE STAMPEDE ke liye REQUEST COALESCING,
      PROBABILISTIC EARLY EXPIRATION, aur HOT-KEY REPLICATION
      lagaunga.

   5. REMAINING database reads ke liye READ REPLICAS use karunga,
      REPLICATION LAG handle karne ke liye READ-YOUR-WRITES aur
      LAG MONITORING lagaunga.

   6. GLOBAL users ke liye CDN use karunga, PUBLIC cacheable
      content ko EDGE pe serve karunga.

   7. Remaining DB queries ko INDEXING aur DENORMALIZATION se
      OPTIMIZE karunga.

   8. EXPENSIVE derived data (jaise feeds, aggregations) ko
      MATERIALIZED VIEWS ya PRE-COMPUTED DATA STORES mein rakhunga.

   9. LOAD BALANCING aur CONNECTION POOLING se traffic evenly
      distribute karunga.

   10. HAR LAYER ka HIT RATE, LATENCY PERCENTILE, aur DB LOAD
       MEASURE karunga taaki optimization data-driven ho."


====================================================================
RAPID FIRE REVISION (Last Minute Ke Liye)
====================================================================

Q: Zyadatar apps READ-HEAVY kyun hote hain?
A: Content EK BAAR create hota hai, MANY BAAR consume hota hai
   (post/video/product) - isliye read:write ratio bahut zyada hota
   hai.

Q: Caching ka sabse important interview metric kya hai?
A: CACHE HIT RATE - sirf "cache lagaya hai" bolna kaafi nahi.

Q: Cache-aside pattern kya hai?
A: Read pe pehle cache check karo, MISS pe DB se lao aur CACHE mein
   store kar do next time ke liye.

Q: Hot key vs Cache stampede mein difference?
A: HOT KEY = ek key pe uneven traffic load. STAMPEDE = jab hot key
   expire hoti hai, sab requests EK SAATH database hit karti hain.

Q: Stampede solve karne ka pehla answer?
A: REQUEST COALESCING (single-flight) - sirf pehli request
   recompute kare, baaki uska result wait karein.

Q: Read replicas kis problem ko SOLVE NAHI karte?
A: SINGLE HOT ROW/KEY problem - saari replicas mein wohi
   contention hoga. Uske liye CACHING chahiye.

Q: Async vs Sync replication mein main trade-off?
A: ASYNC = fast writes, but stale reads (lag). SYNC = durable,
   but slow writes aur availability risk.

Q: Read-your-writes problem kya hai, fix kaise karein?
A: User apna hi WRITE nahi dekh pata (replica lag ki wajah se).
   Fix: recent writers ko THODI DER PRIMARY pe route karo.

Q: CDN kis liye use hota hai - sirf images ke liye?
A: NAHI - static assets + CACHEABLE PUBLIC API responses bhi
   edge pe serve ho sakte hain.

Q: stale-while-revalidate kya karta hai?
A: Expiry ke turant baad bhi EXISTING copy serve karta hai, aur
   BACKGROUND mein refresh karta hai - user kabhi wait nahi karta.

Q: Materialized view refresh karte waqt kya dhyan rakhna hai?
A: Plain REFRESH exclusive lock leta hai, reads block hoti hain.
   REFRESH...CONCURRENTLY use karo, blocking avoid karne ke liye.

Q: Denormalization kab use karein?
A: Jab READ performance critical ho aur redundant data maintain
   karne ki complexity afford ho (analytics, read-heavy views).

Q: Load balancer ke liye least-connections kyun better hai DB
   reads ke liye?
A: Kyunki query durations VARY karti hain - already busy replica
   pe aur load nahi daalna chahiye.

====================================================================
                    END OF INTERVIEW PREP NOTES
   (Practice bolke karo, whiteboard pe layered architecture
    banane ki practice bhi karo - interview mein yehi flow
    follow karna hai)
====================================================================
*/